# Writing a new kernel

Modern attention stacks (Qwen3, OLMo 2, Gemma 3) RMS-normalize every query and key head *before*
rotary embedding. `popcorn.kernels` ships `rms_norm` and `rope` as separate ops, and so do fla and
liger -- nobody fuses the pair. In torch the pair is a chain of elementwise kernels, and every one of
them writes a full-size intermediate back to memory, so `q` and `k` cross HBM many times over.

This notebook builds that fused op, `qk_norm_rope`, end to end: the reference that defines the
semantics, a Triton implementation with a hand-written backward, popcorn's correctness gate, and a
benchmark honest enough to include `torch.compile`.

Nothing here is committed. `register_kernel` mutates the registry of this process only; the last
section covers what turning it into a real contribution takes.

In [1]:
import torch
import triton
import triton.language as tl
from jaxtyping import Float
from torch import Tensor

from popcorn import KERNELS, DispatchError, Tag, register_kernel
from popcorn.bench import report
from popcorn.kernels._utils import rms

torch.manual_seed(0)
print(torch.cuda.get_device_name(), "| triton", triton.__version__, "| torch", torch.__version__)

NVIDIA H100 80GB HBM3 | triton 3.6.0 | torch 2.11.0+cu130


## 1. The reference is the contract

Every popcorn kernel starts as pure torch. The reference defines the semantics, tells the harness how
to build test cases, and is the thing every implementation is graded against -- so it is written for
clarity, never for speed. The jaxtyping annotations name each dimension, which is how the harness
knows `q` and `k` share a `head_dim` but differ in head count (grouped-query attention).

`cos` and `sin` follow the convention `rope` already uses: full head-width tables whose two halves
are duplicates, and non-differentiable, hence the `.detach()`.

In [2]:
KERNELS.pop("qk_norm_rope", None)  # the registry is process-global; keeps this cell re-runnable


def _rotate_half(x):
    x1, x2 = x.chunk(2, -1)
    return torch.cat((-x2, x1), -1)


def _table(f):
    """Test tables duplicate their first half, like a real rope cache."""
    return lambda t: f(t).chunk(2, -1)[0].repeat(1, 1, 2)


@register_kernel(
    test_args={"eps": [1e-6]},
    test_inputs={"cos": _table(torch.cos), "sin": _table(torch.sin)},
    tags={Tag.NORMALIZATION, Tag.POSITIONAL, Tag.FUSED},
)
def qk_norm_rope(
    q: Float[Tensor, "batch q_heads seq head_dim"],
    k: Float[Tensor, "batch kv_heads seq head_dim"],
    q_weight: Float[Tensor, "head_dim"],
    k_weight: Float[Tensor, "head_dim"],
    cos: Float[Tensor, "cos_batch seq head_dim"],
    sin: Float[Tensor, "cos_batch seq head_dim"],
    eps: float = 1e-6,
) -> tuple[Float[Tensor, "batch q_heads seq head_dim"], Float[Tensor, "batch kv_heads seq head_dim"]]:
    r"""Per-head RMS normalization of queries and keys, followed by rotary embedding.

    $$\hat{x} = \frac{x}{\sqrt{\overline{x^2} + \varepsilon}} \odot w, \qquad
      y = \hat{x} \odot \cos_t + \mathrm{rot}_{1/2}(\hat{x}) \odot \sin_t$$

    [Query-Key Normalization (Henry et al., 2020)](https://arxiv.org/abs/2010.04245)
    """
    q, k = rms(q, eps) * q_weight, rms(k, eps) * k_weight
    cos, sin = cos.detach()[:, None], sin.detach()[:, None]
    return q * cos + _rotate_half(q) * sin, k * cos + _rotate_half(k) * sin


qk_norm_rope

qk_norm_rope(q: jaxtyping.Float[Tensor, 'batch q_heads seq head_dim'], k: jaxtyping.Float[Tensor, 'batch kv_heads seq head_dim'], q_weight: jaxtyping.Float[Tensor, 'head_dim'], k_weight: jaxtyping.Float[Tensor, 'head_dim'], cos: jaxtyping.Float[Tensor, 'cos_batch seq head_dim'], sin: jaxtyping.Float[Tensor, 'cos_batch seq head_dim'], eps: float = 1e-06) -> tuple[jaxtyping.Float[Tensor, 'batch q_heads seq head_dim'], jaxtyping.Float[Tensor, 'batch kv_heads seq head_dim']]
  torch

## 2. What the reference costs

Shapes from a Qwen3-8B attention block: 32 query heads, 8 key/value heads, head dimension 128.

Every output element depends on one input element plus its row's mean square, so this op is pure
streaming and its floor is one read and one write of `q` and `k`. Everything above that floor is the
fusion tax: intermediates torch has to materialize because each operator is its own kernel.

In [3]:
def make(B=4, HQ=32, HK=8, T=4096, D=128, grad=True):
    g = dict(device="cuda", dtype=torch.bfloat16)
    half = torch.randn(1, T, D // 2, **g)
    return {
        "q": torch.randn(B, HQ, T, D, **g, requires_grad=grad),
        "k": torch.randn(B, HK, T, D, **g, requires_grad=grad),
        "q_weight": torch.randn(D, **g, requires_grad=grad),
        "k_weight": torch.randn(D, **g, requires_grad=grad),
        "cos": torch.cos(half).repeat(1, 1, 2),
        "sin": torch.sin(half).repeat(1, 1, 2),
    }


inputs = make()
essential = 2 * (inputs["q"].numel() + inputs["k"].numel()) * inputs["q"].element_size()


def bench(fn, inputs):
    """Milliseconds for forward and forward+backward, plus peak GiB of one step."""
    cotangent = [torch.randn_like(o) for o in fn(**inputs)]

    def step():
        for v in inputs.values():
            v.grad = None
        torch.autograd.backward(fn(**inputs), cotangent)

    step()  # warm the backward too: autotuning and lazy compilation must stay out of the timed region
    torch.cuda.synchronize()
    fwd = triton.testing.do_bench(lambda: fn(**inputs), warmup=25, rep=100)
    both = triton.testing.do_bench(step, warmup=25, rep=100)
    torch.cuda.reset_peak_memory_stats()
    step()
    torch.cuda.synchronize()
    return fwd, both, torch.cuda.max_memory_allocated() / 2**30


fwd, both, peak = bench(qk_norm_rope.reference, inputs)
print(f"q {tuple(inputs['q'].shape)}   k {tuple(inputs['k'].shape)}   bf16")
print(f"floor: read q,k once + write once = {essential / 2**20:.0f} MiB")
print(f"eager torch: fwd {fwd:.3f} ms | fwd+bwd {both:.3f} ms | peak {peak:.2f} GiB")
print(f"  -> {essential / (fwd * 1e-3) / 1e12:.2f} TB/s of useful traffic")

q (4, 32, 4096, 128)   k (4, 8, 4096, 128)   bf16
floor: read q,k once + write once = 320 MiB
eager torch: fwd 2.403 ms | fwd+bwd 6.777 ms | peak 2.00 GiB
  -> 0.14 TB/s of useful traffic


## 3. The fused forward

The rotation mixes lane `d` with lane `d + D/2`, so the natural tile is the two halves of a head
vector side by side. With both halves in registers the row's sum of squares, the normalization, and
the rotation all happen without touching memory again: one read, one write.

Two details worth pointing out. `rstd` is stashed for the backward -- one fp32 scalar per row, far
cheaper than saving normalized `q` and `k`. And `cos`/`sin` are loaded at full width instead of
assuming the halves are equal, which keeps the kernel faithful to the reference for any table.

Accumulation is fp32 even though the tensors are bf16. That is not optional: a bf16 sum over 128
squares loses most of its precision.

In [4]:
_CONFIGS = [triton.Config({}, num_warps=w, num_stages=s) for w in (2, 4, 8) for s in (2, 3)]


@triton.autotune(configs=_CONFIGS, key=["D", "BT", "HEADS", "T"])
@triton.jit(do_not_specialize=["T"])
def _fwd(
    X,
    W,
    COS,
    SIN,
    Y,
    RSTD,
    T,
    eps,
    sxb,
    sxh,
    sxt,
    scb,
    sct,
    D: tl.constexpr,
    BD: tl.constexpr,
    BT: tl.constexpr,
    HEADS: tl.constexpr,
):
    i_t, i_h, i_b = tl.program_id(0), tl.program_id(1), tl.program_id(2)
    R: tl.constexpr = D // 2
    o_t = i_t * BT + tl.arange(0, BT)
    o_d = tl.arange(0, BD)
    m_t, m_d = o_t < T, o_d < R
    mask = m_t[:, None] & m_d[None, :]

    p_x = X + i_b * sxb + i_h * sxh + o_t[:, None] * sxt + o_d[None, :]
    x0 = tl.load(p_x, mask=mask, other=0.0).to(tl.float32)
    x1 = tl.load(p_x + R, mask=mask, other=0.0).to(tl.float32)

    # Both halves are resident, so the row reduction needs no second pass.
    r = tl.rsqrt((tl.sum(x0 * x0, 1) + tl.sum(x1 * x1, 1)) / D + eps)
    tl.store(RSTD + (i_b * HEADS + i_h) * T + o_t, r, mask=m_t)

    w0 = tl.load(W + o_d, mask=m_d, other=0.0).to(tl.float32)
    w1 = tl.load(W + o_d + R, mask=m_d, other=0.0).to(tl.float32)
    n0 = x0 * r[:, None] * w0[None, :]
    n1 = x1 * r[:, None] * w1[None, :]

    p_c = COS + i_b * scb + o_t[:, None] * sct + o_d[None, :]
    p_s = SIN + i_b * scb + o_t[:, None] * sct + o_d[None, :]
    c0 = tl.load(p_c, mask=mask, other=1.0).to(tl.float32)
    c1 = tl.load(p_c + R, mask=mask, other=1.0).to(tl.float32)
    s0 = tl.load(p_s, mask=mask, other=0.0).to(tl.float32)
    s1 = tl.load(p_s + R, mask=mask, other=0.0).to(tl.float32)

    p_y = Y + i_b * sxb + i_h * sxh + o_t[:, None] * sxt + o_d[None, :]
    tl.store(p_y, n0 * c0 - n1 * s0, mask=mask)
    tl.store(p_y + R, n1 * c1 + n0 * s1, mask=mask)

## 4. The backward recomputes instead of saving

Three gradients to derive, working backwards through the op.

The rotation is linear, so its gradient is its transpose. With $y_0 = n_0 c_0 - n_1 s_0$ and
$y_1 = n_1 c_1 + n_0 s_1$, the normalized halves get

$$g_0 = \bar{y}_0 c_0 + \bar{y}_1 s_1, \qquad g_1 = \bar{y}_1 c_1 - \bar{y}_0 s_0$$

Then RMSNorm. With $u = g \odot w$ and $r$ the saved reciprocal norm, the input gradient carries the
usual projection term, because $r$ itself depends on every element of the row:

$$\bar{x} = r\left(u - \frac{x \, r^2 \langle u, x \rangle}{D}\right)$$

The weight gradient $\bar{w} = \sum_{\text{rows}} g \odot x \, r$ is a reduction over every row
in the batch, which no single program can own. Each program writes one partial row and the caller
sums them.

Nothing from the forward is reloaded except `x` and `rstd`: the normalized values are recomputed in
registers. That is what keeps peak memory below the compiled version, which saves them instead.

In [5]:
@triton.autotune(configs=_CONFIGS, key=["D", "BT", "HEADS", "T"])
@triton.jit(do_not_specialize=["T"])
def _bwd(
    X,
    W,
    COS,
    SIN,
    DY,
    DX,
    DW,
    RSTD,
    T,
    sxb,
    sxh,
    sxt,
    scb,
    sct,
    D: tl.constexpr,
    BD: tl.constexpr,
    BT: tl.constexpr,
    HEADS: tl.constexpr,
):
    i_t, i_h, i_b = tl.program_id(0), tl.program_id(1), tl.program_id(2)
    R: tl.constexpr = D // 2
    o_t = i_t * BT + tl.arange(0, BT)
    o_d = tl.arange(0, BD)
    m_t, m_d = o_t < T, o_d < R
    mask = m_t[:, None] & m_d[None, :]

    base = i_b * sxb + i_h * sxh + o_t[:, None] * sxt + o_d[None, :]
    p_x, p_dy = X + base, DY + base
    x0 = tl.load(p_x, mask=mask, other=0.0).to(tl.float32)
    x1 = tl.load(p_x + R, mask=mask, other=0.0).to(tl.float32)
    dy0 = tl.load(p_dy, mask=mask, other=0.0).to(tl.float32)
    dy1 = tl.load(p_dy + R, mask=mask, other=0.0).to(tl.float32)

    p_c = COS + i_b * scb + o_t[:, None] * sct + o_d[None, :]
    p_s = SIN + i_b * scb + o_t[:, None] * sct + o_d[None, :]
    c0 = tl.load(p_c, mask=mask, other=1.0).to(tl.float32)
    c1 = tl.load(p_c + R, mask=mask, other=1.0).to(tl.float32)
    s0 = tl.load(p_s, mask=mask, other=0.0).to(tl.float32)
    s1 = tl.load(p_s + R, mask=mask, other=0.0).to(tl.float32)

    # Transpose of the rotation.
    g0 = dy0 * c0 + dy1 * s1
    g1 = dy1 * c1 - dy0 * s0

    w0 = tl.load(W + o_d, mask=m_d, other=0.0).to(tl.float32)
    w1 = tl.load(W + o_d + R, mask=m_d, other=0.0).to(tl.float32)
    r = tl.load(RSTD + (i_b * HEADS + i_h) * T + o_t, mask=m_t, other=0.0)

    u0, u1 = g0 * w0[None, :], g1 * w1[None, :]
    dot = tl.sum(u0 * x0, 1) + tl.sum(u1 * x1, 1)
    coef = r * r * r * dot / D
    p_dx = DX + base
    tl.store(p_dx, r[:, None] * u0 - x0 * coef[:, None], mask=mask)
    tl.store(p_dx + R, r[:, None] * u1 - x1 * coef[:, None], mask=mask)

    # dw spans every row, so each program leaves one partial for the caller to sum.
    pid = (i_b * HEADS + i_h) * tl.num_programs(0) + i_t
    tl.store(DW + pid * D + o_d, tl.sum(g0 * x0 * r[:, None], 0), mask=m_d)
    tl.store(DW + pid * D + o_d + R, tl.sum(g1 * x1 * r[:, None], 0), mask=m_d)

Wiring it up. One `autograd.Function` handles a single tensor and its weight; the adapter applies it
to `q` and `k`. `cos`, `sin`, and `eps` get `None` gradients, matching the reference's `.detach()`.

An adapter's parameters must match the reference's exactly, which is what lets the harness call every
implementation the same way.

In [6]:
BT = 32  # flat from 8 to 64: the kernel is bandwidth-bound, not occupancy-bound


def _strides(x, cos):
    # A one-batch cos broadcasts across the batch; stride 0 says so.
    return (x.stride(0), x.stride(1), x.stride(2), cos.stride(0) if cos.shape[0] > 1 else 0, cos.stride(-2))


class _NormRope(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, w, cos, sin, eps):
        x, cos, sin = x.contiguous(), cos.contiguous(), sin.contiguous()
        B, H, T, D = x.shape
        y = torch.empty_like(x)
        rstd = torch.empty(B, H, T, device=x.device, dtype=torch.float32)
        _fwd[(triton.cdiv(T, BT), H, B)](
            x,
            w,
            cos,
            sin,
            y,
            rstd,
            T,
            eps,
            *_strides(x, cos),
            D=D,
            BD=triton.next_power_of_2(D // 2),
            BT=BT,
            HEADS=H,
        )
        ctx.save_for_backward(x, w, cos, sin, rstd)
        return y

    @staticmethod
    def backward(ctx, dy):
        x, w, cos, sin, rstd = ctx.saved_tensors
        B, H, T, D = x.shape
        blocks = triton.cdiv(T, BT)
        dx = torch.empty_like(x)
        partial = torch.empty(B * H * blocks, D, device=x.device, dtype=torch.float32)
        _bwd[(blocks, H, B)](
            x,
            w,
            cos,
            sin,
            dy.contiguous(),
            dx,
            partial,
            rstd,
            T,
            *_strides(x, cos),
            D=D,
            BD=triton.next_power_of_2(D // 2),
            BT=BT,
            HEADS=H,
        )
        return dx, partial.sum(0).to(w.dtype), None, None, None


@qk_norm_rope.register("popcorn")
def qk_norm_rope_popcorn(q, k, q_weight, k_weight, cos, sin, eps):
    return _NormRope.apply(q, q_weight, cos, sin, eps), _NormRope.apply(k, k_weight, cos, sin, eps)


qk_norm_rope.available_impls()

('popcorn', 'torch')

## 5. The correctness gate

`op.validate` builds a case from the annotations, runs it against a float64 ground truth, and grades
the error against the *reference's own* error at the native dtype. A kernel is allowed to be
inaccurate only where torch already is, so there is no hand-picked tolerance to argue about -- and
being more accurate than the reference, which this kernel is because it keeps fp32 in registers,
never counts against it.

In [7]:
small = make(B=2, HQ=8, HK=2, T=1024)
for record in qk_norm_rope.validate(**small):
    report(record)

popcorn  pass
         err 3.06e-02 on scale 11.4


`benchmark` runs the same comparison and times it. Small shapes are dominated by launch overhead, so
this understates the win -- the next section uses realistic ones.

In [8]:
for record in qk_norm_rope.benchmark(**small):
    report(record)

popcorn  torch      0.1 ms -> popcorn    0.0 ms   (4.4x)
         peak         0.28 GB ->      0.25 GB
         bwd           0.4 ms ->    0.2 ms   (2.2x)
         err 3.06e-02 on scale 11.4


## 6. Evidence decides what the kernel may serve

Those two calls are now the only proof this kernel works, and popcorn takes that literally. Asking the
dispatcher to *force* the new implementation onto the larger shape fails, because 8 key/value heads is
outside the region the evidence covers:

In [9]:
try:
    qk_norm_rope(**inputs, impl="popcorn")
except DispatchError as error:
    print(f"DispatchError: {error}")

DispatchError: qk_norm_rope: implementation 'popcorn' is outside its validity region: q_heads=32 outside [8, 8]


Without `impl=`, there is no error and no silence either -- the call is simply served by the reference,
because an unproven implementation never wins a dispatch. This is the mechanism that keeps a kernel
that was only ever checked at one shape from quietly serving every other shape, and running the real
grid is what widens the region.

It also means the benchmark below has to call the implementation function directly. That is the honest
thing to measure anyway: the question is how fast the kernel is, not how fast dispatch is.

## 7. Is it actually fast?

The grid compares implementations against the eager reference, which is the right contract but a soft
target. The harder question is whether a hand-written kernel still earns its place next to inductor,
so `torch.compile` goes in the table too.

In [10]:
def composed(q, k, q_weight, k_weight, cos, sin):
    """Unfused, but every piece is already a tuned popcorn kernel."""
    from popcorn.kernels import rms_norm, rope

    return rope(rms_norm(q, q_weight), rms_norm(k, k_weight), cos, sin)


variants = {
    "torch eager": qk_norm_rope.reference,
    "torch.compile": torch.compile(qk_norm_rope.reference, fullgraph=True),
    "popcorn unfused": composed,
    "fused triton": lambda **kw: qk_norm_rope_popcorn(**kw, eps=1e-6),
}

rows = {}
print(f"{'variant':16s} {'fwd':>10s} {'fwd+bwd':>10s} {'peak':>9s} {'bwd only':>10s}")
for name, fn in variants.items():
    fn(**inputs)  # pay compilation and autotuning outside the timed region
    rows[name] = bench(fn, inputs)
    f, b, p = rows[name]
    print(f"{name:16s} {f:7.3f} ms {b:7.3f} ms {p:6.2f} GiB {b - f:7.3f} ms")

variant                 fwd    fwd+bwd      peak   bwd only


torch eager        2.402 ms   6.781 ms   2.01 GiB   4.379 ms


torch.compile      0.141 ms   0.481 ms   0.77 GiB   0.341 ms


popcorn unfused    1.183 ms   2.499 ms   0.85 GiB   1.316 ms


fused triton       0.127 ms   0.401 ms   0.66 GiB   0.273 ms


## 8. Reading the result honestly

Against eager torch the fused kernel is worth roughly 19x on the forward, and that is the number most
users actually feel, because most code is eager.

Against `torch.compile` the forward is a near tie, and that is the interesting part. Both are at the
memory roofline, so neither can win: the op moves a fixed amount of data and the hardware moves it at
a fixed rate. Inductor is very good at exactly this kind of elementwise chain, and a hand-written
kernel does not beat physics.

Where the fused version still wins is the backward and peak memory, for one reason: it recomputes the
normalized values from `x` and `rstd` instead of saving them. Every variant's effective bandwidth is
below, with the practical ceiling for a streaming read+write being roughly 80% of the spec sheet.

In [11]:
peak_tbs = {"NVIDIA H100 80GB HBM3": 3.35, "NVIDIA A100-SXM4-80GB": 2.04}.get(torch.cuda.get_device_name())
print(f"useful traffic per forward: {essential / 2**20:.0f} MiB")
for name, (f, b, _) in rows.items():
    share = f" ({essential / (f * 1e-3) / 1e12 / peak_tbs:.0%} of HBM peak)" if peak_tbs else ""
    print(f"  {name:16s} {essential / (f * 1e-3) / 1e12:5.2f} TB/s{share}")
if peak_tbs:
    print(f"\n{torch.cuda.get_device_name()} HBM peak: {peak_tbs:.2f} TB/s")

useful traffic per forward: 320 MiB
  torch eager       0.14 TB/s (4% of HBM peak)
  torch.compile     2.38 TB/s (71% of HBM peak)
  popcorn unfused   0.28 TB/s (8% of HBM peak)
  fused triton      2.63 TB/s (79% of HBM peak)

NVIDIA H100 80GB HBM3 HBM peak: 3.35 TB/s


## 9. Turning this into a contribution

The notebook version is self-contained on purpose. A real kernel is split in two:

- `src/popcorn/kernels/qk_norm_rope.py` -- the reference plus adapters, one per implementation.
- `src/popcorn/impls/qk_norm_rope_tl.py` -- the Triton code, imported lazily through `source=`.

Then it needs evidence. Shape validity in popcorn comes from benchmark reports rather than hand-written
claims, so the op has to run the real grid on real hardware before any performance number is quoted:

```bash
uv run pytest tests -q
uv run pyright && scripts/format.sh
uv run python -m popcorn.bench run qk_norm_rope --impl popcorn
```

Every row must come back without `fail`, `crash`, `error`, or `bench_error`, across the dtypes and
shapes the grid generates -- this notebook only ever exercised bf16 at two shapes on one GPU.
[CONTRIBUTING.md](../CONTRIBUTING.md) is the source of truth for the rest.

Two things this kernel does *not* do, worth knowing before copying it: tuple-returning ops get no
`torch.ops.popcorn` binding, so they are invisible to `popcorn.compile` rewrites, and the reference's
duplicated-halves `cos`/`sin` convention is inherited from `rope` rather than chosen here.